In [2]:
from spiral import Spiral
import pyarrow as pa

sp = Spiral()
project = sp.project("external-805943")

# Frame dimensions: 236 x 420 x 3 = 297360 float16 values
FRAME_SIZE = 236 * 420 * 3

# Random IDs are discouraged; use monotonically increasing IDs where possible, like UUIDv7.
experiments_table = project.create_table("experiments", key_schema=pa.schema({"data_key": pa.string()}), exist_ok=True)
eye_trackers_table = project.create_table("eye-trackers-v2", key_schema=pa.schema({"data_key": pa.string(), "step": pa.int64()}), exist_ok=True)
responses_table = project.create_table("responses-v2", key_schema=pa.schema({"data_key": pa.string(), "step": pa.int64()}), exist_ok=True)
screens_table = project.create_table("screens-v2", key_schema=pa.schema({"data_key": pa.string(), "step": pa.int64()}), exist_ok=True)

In [3]:
from pathlib import Path
import yaml
import json

# Loads experiment data from a directory.
experiment_data_path = Path("35_3832003544063_1_V1")

with open(experiment_data_path / "meta.json", "r") as f:
    global_meta = json.load(f)

In [4]:
import pyarrow as pa
import numpy as np

def constant_array(value, length, pa_type):
    """
    Create a constant array in PyArrow with optimal memory efficiency.

    Parameters:
    -----------
    value : any
        The constant value to repeat
    length : int
        The length of the array
    pa_type : pa.DataType
        The PyArrow data type

    Returns:
    --------
    pa.Array
        A constant array with the specified value repeated
    """
    # Create a scalar with the specified type
    scalar = pa.scalar(value, type=pa_type)

    # Use repeat to create a constant array efficiently
    # This uses run-length encoding internally for maximum efficiency
    return pa.repeat(scalar, length)

# Example: Convert (X, Y) numpy array to PyArrow list array of length X
def numpy_to_pyarrow_list(arr, pa_type):
    """
    Convert numpy array of shape (X, Y) to PyArrow list array of length X.
    Each element in the list array contains a Y-length array.
    """
    # Create PyArrow array from the numpy array (flattened view)
    values = pa.array(arr.ravel(), type=pa_type)

    # Create offsets: [0, Y, 2*Y, 3*Y, ..., X*Y]
    offsets = pa.array(np.arange(0, arr.size + 1, arr.shape[1]), type=pa.int32())

    # Construct the list array
    list_array = pa.ListArray.from_arrays(offsets, values)

    return list_array

In [5]:
eye_tracker_means = np.load(experiment_data_path / "eye_tracker" / "meta" / "means.npy")
eye_tracker_stds = np.load(experiment_data_path / "eye_tracker" / "meta" / "stds.npy")
with open(experiment_data_path / "eye_tracker" / "meta.yml", "r") as f:
    eye_tracker_meta = yaml.safe_load(f)

eye_tracker_data = np.memmap(experiment_data_path / "eye_tracker" / "data.mem",
                             dtype=eye_tracker_meta["dtype"],
                             mode="r",
                             shape=(eye_tracker_meta["n_timestamps"], eye_tracker_meta["n_signals"]),)
eye_tracker_time = np.linspace(eye_tracker_meta["start_time"], eye_tracker_meta["end_time"], eye_tracker_meta["n_timestamps"], endpoint=False, dtype=np.float32)


In [ ]:
eye_trackers_table.write({
    "data_key": constant_array(global_meta["data_key"], eye_tracker_meta["n_timestamps"], pa.string()),
    "step": pa.array(range(eye_tracker_meta["n_timestamps"]), type=pa.int64()),
    "time": pa.array(eye_tracker_time, type=pa.float32()),
    # This can also be a list in a single column.
    "x": pa.array(eye_tracker_data[:, 0], type=pa.float32()),
    "y": pa.array(eye_tracker_data[:, 1], type=pa.float32()),
    "pupil": pa.array(eye_tracker_data[:, 2], type=pa.float32()),
})

In [7]:
responses_means = np.load(experiment_data_path / "responses_30Hz_no_filtering" / "meta" / "means.npy")
responses_stds = np.load(experiment_data_path / "responses_30Hz_no_filtering" / "meta" / "stds.npy")
with open(experiment_data_path / "responses_30Hz_no_filtering" / "meta.yml", "r") as f:
    responses_meta = yaml.safe_load(f)

responses_data = np.memmap(experiment_data_path / "responses_30Hz_no_filtering" / "data.mem",
                           dtype=responses_meta["dtype"],
                           mode="r",
                           shape=(responses_meta["n_timestamps"], responses_meta["n_signals"]),)
responses_time = np.linspace(responses_meta["start_time"], responses_meta["end_time"], responses_meta["n_timestamps"], endpoint=False, dtype=np.float32)
signals = numpy_to_pyarrow_list(responses_data, pa.float32())

In [7]:
responses_table.write({
    "data_key": constant_array(global_meta["data_key"], responses_meta["n_timestamps"], pa.string()),
    "step": pa.array(range(responses_meta["n_timestamps"]), type=pa.int64()),
    "time": pa.array(responses_time, type=pa.float32()),
    "signals": pa.array(signals, type=pa.list_(pa.float32())),
})

2026-01-12T16:42:21.637707Z  WARN HttpMetastore::list_column_groups: spiral_api: SpiralDB is using a deprecated API endpoint, please migrate to a supported version path="table_m5tl10/column-groups" deprecation_date=2025-12-17 00:00:00 UTC sunset_date=unknown
2026-01-12T16:42:27.063804Z  INFO transaction.commit: spiral_table::transaction: Transaction committed successfully table_id=table_m5tl10 operation_count=3 retry_attempt=0


In [8]:
timestamps = np.load(experiment_data_path / "screen" / "timestamps.npy")
with open(experiment_data_path / "screen" / "meta.yml", "r") as f:
    screen_meta = yaml.safe_load(f)

screen_data_meta = []
for i, frame_meta_path in enumerate(sorted((experiment_data_path / "screen" / "meta").glob("*.yml"))):
    with open(frame_meta_path, "r") as f:
        screen_data_meta.append(yaml.safe_load(f))

print(f"Loaded metadata for {len(screen_data_meta)} frames")

Loaded metadata for 6098 frames


In [39]:
# Write metadata into experiments table
experiment = dict(global_meta)
experiment["eye_tracker"] = dict(eye_tracker_meta)
experiment["eye_tracker"]["means"] = eye_tracker_means
experiment["eye_tracker"]["stds"] = eye_tracker_stds
experiment["responses"] = dict(responses_meta)
experiment["responses"]["means"] = responses_means
experiment["responses"]["stds"] = responses_stds
experiment["screen"] = dict(screen_meta)
experiment["screen"]["metas"] = screen_data_meta
experiment["screen"]["timestamps"] = timestamps

In [40]:
experiments_table.write([experiment])

2026-01-08T15:52:22.569221Z  INFO transaction.commit: spiral_table::transaction: Transaction committed successfully table_id=table_cdzw2p operation_count=29 retry_attempt=0


In [10]:
import itertools
from concurrent.futures import ThreadPoolExecutor
from spiral.core.table.spec import Operation


def process_batch(batch_data):
    """Process a single batch and return transaction ops."""
    batch, data_key = batch_data

    # Create a new transaction for this worker
    worker_tx = screens_table.txn()

    screen_step = []
    screen_time = []
    screen_data = []
    for i, frame_path in batch:
        # NOTE(marko): This is weird, but some frame data is missing...
        #   so we can't just take screen_data_meta[i].
        index = int(str(frame_path).split("/")[-1].strip(".npy"))
        meta = screen_data_meta[index]
        timestamps_offset = meta["first_frame_idx"]
        data = np.load(frame_path)
        assert data.shape[0] == meta["num_frames"], f"Data shape {data.shape} does not match metadata {meta}"
        for j in range(meta["num_frames"]):
            screen_step.append(timestamps_offset + j)
            screen_time.append(timestamps[timestamps_offset + j])
            screen_data.append(data[j, :, :, :].ravel())

    # Convert list of numpy arrays to PyArrow fixed-size list array
    flat_values = pa.array(np.concatenate(screen_data), type=pa.float16())
    frame_array = pa.FixedSizeListArray.from_arrays(flat_values, FRAME_SIZE)

    worker_tx.write({
        "data_key": constant_array(data_key, len(screen_time), pa.string()),
        "step": pa.array(screen_step, type=pa.int64()),
        "time": pa.array(screen_time, type=pa.float32()),
        "frame": frame_array,
    })

    # Take operations aborting the transaction. We want all workers to atomically commit.
    return [op.to_json() for op in worker_tx.take()]


# Prepare batches
batch_size = 100
batches = list(itertools.batched(enumerate(sorted((experiment_data_path / "screen" / "data").glob("*.npy"))), batch_size))

# Prepare batch data for workers
batch_data_list = [
    (batch, global_meta["data_key"])
    for batch in batches
]

# Create root transaction
tx = screens_table.txn()

# Process batches with thread pool (avoids pickle issues in notebooks)
with ThreadPoolExecutor(max_workers=8) as executor:
    all_ops = list(executor.map(process_batch, batch_data_list))

# Add all ops to the root transaction
for ops in all_ops:
    tx.include([Operation.from_json(op) for op in ops])

# Commit the root transaction
tx.commit()

2026-01-12T17:51:31.646720Z  WARN HttpMetastore::list_column_groups: spiral_api: SpiralDB is using a deprecated API endpoint, please migrate to a supported version path="table_8zdjtk/column-groups" deprecation_date=2025-12-17 00:00:00 UTC sunset_date=unknown
2026-01-12T17:51:31.798185Z  WARN HttpMetastore::list_column_groups: spiral_api: SpiralDB is using a deprecated API endpoint, please migrate to a supported version path="table_8zdjtk/column-groups" deprecation_date=2025-12-17 00:00:00 UTC sunset_date=unknown
2026-01-12T17:51:31.814126Z  WARN HttpMetastore::list_column_groups: spiral_api: SpiralDB is using a deprecated API endpoint, please migrate to a supported version path="table_8zdjtk/column-groups" deprecation_date=2025-12-17 00:00:00 UTC sunset_date=unknown
2026-01-12T17:51:31.814145Z  WARN HttpMetastore::list_column_groups: spiral_api: SpiralDB is using a deprecated API endpoint, please migrate to a supported version path="table_8zdjtk/column-groups" deprecation_date=2025-12-